# Lecture 8 — Reference Genome Indexing, STAR Alignment & Post-Alignment QC

Align sequencing reads to a reference genome with **STAR**, on free Google Colab.

| Field | Value |
|---|---|
| **Dataset** | GSE129221 |
| **Sample** | `GSM3702795` ("A1 PC9 cells") |
| **Reads** | 5,000 pairs × 150 bp |
| **Reference** | GRCh38 (Ensembl release 115) |

**What we produce** — a raw-vs-trimmed FastQC/MultiQC report, a sorted BAM, and alignment QC.


## 0. Prerequisites

### Upload the data

Download **`lecture-08-data.tar.gz`** (56 MB):

[Google Drive — lecture-08-data.tar.gz](https://drive.google.com/file/d/1RCf4kJOXrEvZdOjLWc18GdRdHdcq6D2S/view?usp=sharing)

In Colab, open the **Files** panel in the left sidebar (`⋮` → *Upload*), or just drag the file onto the notebook. It should land at:

```
/content/lecture-08-data.tar.gz
```

Check it arrived whole before unpacking — a partial upload fails halfway through and leaves a broken `data/`:

```bash
stat -c%s /content/lecture-08-data.tar.gz     # expect 57779840
```

Then run the next cell to unpack.

In [1]:
%%bash

cd /content
# Unpack
tar -xzf lecture-08-data.tar.gz
du -sh data/*

287M	data/gtf
3.2M	data/raw_fastq
176M	data/ref


### Installing the tools

Colab ships none of the bioinformatics tools, and there is no conda or micromamba on the image. We install from Ubuntu's `apt` archive and PyPI:

| package | provides | version |
|---|---|---|
| `rna-star` | STAR | 2.7.11b |
| `samtools` | samtools | 1.19.2 |
| `subread` | featureCounts | 2.0.6 |
| `fastqc` | FastQC | 0.11.9 |
| `fastp` | fastp | 0.23.2 |
| `multiqc` | MultiQC | pip (latest) |

In [2]:
%%bash

apt-get update -qq
apt-get install -y -qq rna-star samtools subread fastqc fastp
pip install -q multiqc

STAR --version
fastqc --version
fastp --version
multiqc --version

Selecting previously unselected package at-spi2-common.
(Reading database ... 122806 files and directories currently installed.)
Preparing to unpack .../00-at-spi2-common_2.52.0-1build1_all.deb ...
Unpacking at-spi2-common (2.52.0-1build1) ...
Selecting previously unselected package libatspi2.0-0t64:amd64.
Preparing to unpack .../01-libatspi2.0-0t64_2.52.0-1build1_amd64.deb ...
Unpacking libatspi2.0-0t64:amd64 (2.52.0-1build1) ...
Selecting previously unselected package libxtst6:amd64.
Preparing to unpack .../02-libxtst6_2%3a1.2.3-1.1ubuntu0.1_amd64.deb ...
Unpacking libxtst6:amd64 (2:1.2.3-1.1ubuntu0.1) ...
Selecting previously unselected package session-migration.
Preparing to unpack .../03-session-migration_0.3.9build1_amd64.deb ...
Unpacking session-migration (0.3.9build1) ...
Selecting previously unselected package gsettings-desktop-schemas.
Preparing to unpack .../04-gsettings-desktop-schemas_46.1-0ubuntu1_all.deb ...
Unpacking gsettings-desktop-schemas (46.1-0ubuntu1) ...
Select

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
fastp 0.23.4


### Pre-flight

Before spending 25 minutes on an index, prove the inputs are intact. Two checks:

1. **Mate counts match.** Both files must have the same number of records, divisible by 4.
2. **The first record is readable.** Header, sequence, `+`, quality — four lines.

If either check fails, fix it now. STAR will not fix it for you.


In [3]:
%%bash

cd /content

echo "=== mate counts ==="
for f in data/raw_fastq/*.sub.fq; do
  n=$(wc -l < "$f")
  echo "$(basename "$f"): $((n / 4)) reads"
done

echo
echo "=== first record ==="
head -4 data/raw_fastq/SRR8836894_R1.sub.fq


=== mate counts ===
SRR8836894_R1.sub.fq: 5000 reads
SRR8836894_R2.sub.fq: 5000 reads

=== first record ===
@SRR8836894.1 1/1
GTCGGAGCACATCTTGCTGGCCCAGGTGTTCAGCAAGATGGAGCAGGTGTCTGGCGTGCTGGGCATCGAGGACTACTCGGTCAGCCAGACCACACTGGACAATGTGTTCGTGCACTTTGCCAAGATGCAGAGTGACCACTTTGAGAAGCA
+
--AFA<FFF77<AF-AAF7FAAJJJ7JFAJ-FFF-FF--7---F<<F<AJJFFAFJJAJ-FJF-7<A-<A7-7--<AJJJJJFJ7-7FJAA-F-7AAJ-7-77AA7FFJA--7AFJ<7<---7F--777A7F--A-------7-F--A-7


### FastQC on the raw reads

Before trimming, measure the raw reads. FastQC writes one HTML report and one `.zip` per FASTQ:

```bash
fastqc -t 2 -o data/fastqc_raw data/raw_fastq/*.sub.fq
```

Open either HTML from the Files panel. The report is a set of alarms, not a score — read the per-base quality, adapter content, and overrepresented sequences.

In [4]:
%%bash

cd /content

mkdir -p data/fastqc_raw

# -t 2: two threads (Colab has 2 vCPU)
# -o data/fastqc_raw: HTML + .zip report directory
# data/raw_fastq/*.sub.fq: both raw mate FASTQ files
fastqc \
  -t 2 \
  -o data/fastqc_raw \
  data/raw_fastq/*.sub.fq

ls -lh data/fastqc_raw/

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
null
null
Analysis complete for SRR8836894_R1.sub.fq
Analysis complete for SRR8836894_R2.sub.fq
total 1.9M
-rw-r--r-- 1 root root 579K Oct 25 13:29 SRR8836894_R1.sub_fastqc.html
-rw-r--r-- 1 root root 369K Oct 25 13:29 SRR8836894_R1.sub_fastqc.zip
-rw-r--r-- 1 root root 571K Oct 25 13:29 SRR8836894_R2.sub_fastqc.html
-rw-r--r-- 1 root root 348K Oct 25 13:29 SRR8836894_R2.sub_fastqc.zip


Started analysis of SRR8836894_R1.sub.fq
Approx 20% complete for SRR8836894_R1.sub.fq
Approx 40% complete for SRR8836894_R1.sub.fq
Approx 60% complete for SRR8836894_R1.sub.fq
Approx 80% complete for SRR8836894_R1.sub.fq
Approx 100% complete for SRR8836894_R1.sub.fq
Started analysis of SRR8836894_R2.sub.fq
Approx 20% complete for SRR8836894_R2.sub.fq
Approx 40% complete for SRR8836894_R2.sub.fq
Approx 60% complete for SRR8836894_R2.sub.fq
Approx 80% complete for SRR8836894_R2.sub.fq
Approx 100% complete for SRR8836894_R2.sub.fq


### Trim with fastp

`fastp` removes adapters and low-quality 3′ tails while keeping the read:

- `--detect_adapter_for_pe` — detect adapters from the read pairs themselves.
- `-q 20` — trim the 3′ end where quality drops below Q20.
- `-l 36` — discard reads shorter than 36 bp after trimming.

Both mates stay synchronized: if one read becomes too short, fastp drops the pair.

### Read it in three questions

1. **X-axis?** Position in the read — reads have directionality, not genomic coordinates.
2. **Biology or chemistry?** 3′ decay is chemistry; adapter read-through means a short insert.
3. **One action?** Trim adapters + tail. Nothing else needs fixing.

> **QC is a decision, not a decoration.** The report exists to choose an action, not to fill a methods paragraph.

### What the report showed

- Quality is high across most of the read.
- The tail is short; the adapter signal is small (~2%).
- GC is a single peak at 50% — no contamination signature.

> **Verdict:** trim adapters and the 3′ tail; leave the rest.

In [5]:
%%bash

cd /content

mkdir -p data/trimmed data/fastp

# Inputs: raw paired-end FASTQ
# Outputs: trimmed paired-end FASTQ
# Reports: fastp HTML + JSON inside data/fastp/
fastp \
  -i data/raw_fastq/SRR8836894_R1.sub.fq \
  -I data/raw_fastq/SRR8836894_R2.sub.fq \
  -o data/trimmed/SRR8836894_R1.trim.fq \
  -O data/trimmed/SRR8836894_R2.trim.fq \
  --detect_adapter_for_pe \
  -q 20 \
  -l 36 \
  -h data/fastp/fastp.html \
  -j data/fastp/fastp.json \
  --thread 2

Detecting adapter sequence for read1...
No adapter detected for read1

Detecting adapter sequence for read2...
No adapter detected for read2

Read1 before filtering:
total reads: 5000
total bases: 750000
Q20 bases: 698025(93.07%)
Q30 bases: 632328(84.3104%)

Read2 before filtering:
total reads: 5000
total bases: 750000
Q20 bases: 724039(96.5385%)
Q30 bases: 686744(91.5659%)

Read1 after filtering:
total reads: 4944
total bases: 736405
Q20 bases: 686903(93.2779%)
Q30 bases: 623368(84.6502%)

Read2 after filtering:
total reads: 4944
total bases: 736405
Q20 bases: 715349(97.1407%)
Q30 bases: 680364(92.3899%)

Filtering result:
reads passed filter: 9888
reads failed due to low quality: 112
reads failed due to too many N: 0
reads failed due to too short: 0
reads with adapter trimmed: 434
bases trimmed due to adapters: 10420

Duplication rate: 1.54%

Insert size peak (evaluated by paired-end reads): 262

JSON report: data/fastp/fastp.json
HTML report: data/fastp/fastp.html

fastp -i data/raw

### Check the trimmed reads

Both mates must still have the same number of records before alignment.

In [6]:
%%bash

cd /content

echo "=== trimmed mate counts ==="
for f in data/trimmed/*.trim.fq; do
  n=$(wc -l < "$f")
  echo "$(basename "$f"): $((n / 4)) reads"
done

=== trimmed mate counts ===
SRR8836894_R1.trim.fq: 4944 reads
SRR8836894_R2.trim.fq: 4944 reads


### FastQC on trimmed reads

Run FastQC again on the trimmed FASTQ. This report is the **after** half of the raw-vs-trimmed comparison.

In [7]:
%%bash

cd /content

mkdir -p data/fastqc_trimmed

# -t 2: two threads (Colab has 2 vCPU)
# -o data/fastqc_trimmed: HTML + .zip report directory
# data/trimmed/*.trim.fq: both trimmed mate FASTQ files
fastqc \
  -t 2 \
  -o data/fastqc_trimmed \
  data/trimmed/*.trim.fq

ls -lh data/fastqc_trimmed/

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
null
null
Analysis complete for SRR8836894_R1.trim.fq
Analysis complete for SRR8836894_R2.trim.fq
total 1.9M
-rw-r--r-- 1 root root 585K Oct 25 13:32 SRR8836894_R1.trim_fastqc.html
-rw-r--r-- 1 root root 354K Oct 25 13:32 SRR8836894_R1.trim_fastqc.zip
-rw-r--r-- 1 root root 585K Oct 25 13:32 SRR8836894_R2.trim_fastqc.html
-rw-r--r-- 1 root root 342K Oct 25 13:32 SRR8836894_R2.trim_fastqc.zip


Started analysis of SRR8836894_R1.trim.fq
Approx 20% complete for SRR8836894_R1.trim.fq
Approx 40% complete for SRR8836894_R1.trim.fq
Approx 60% complete for SRR8836894_R1.trim.fq
Started analysis of SRR8836894_R2.trim.fq
Approx 80% complete for SRR8836894_R1.trim.fq
Approx 20% complete for SRR8836894_R2.trim.fq
Approx 40% complete for SRR8836894_R2.trim.fq
Approx 60% complete for SRR8836894_R2.trim.fq
Approx 80% complete for SRR8836894_R2.trim.fq


### Create the MultiQC config

This YAML keeps the raw and trimmed FastQC reports in separate sections.

In [8]:
%%bash

cd /content

mkdir -p data/multiqc

cat > data/multiqc_config.yaml <<'EOF'
title: "Lecture 08 — Raw vs Trimmed Reads"
subtitle: "FastQC before and after fastp"
intro_text: "FastQC reports are grouped separately for the raw 5,000-pair subset and the fastp-trimmed reads."

module_order:
  - fastqc:
      name: "FastQC — raw"
      path_filters:
        - "fastqc_raw/*"
  - fastqc:
      name: "FastQC — trimmed"
      path_filters:
        - "fastqc_trimmed/*"
  - fastp:
      name: "fastp — trimming summary"
EOF

cat data/multiqc_config.yaml

title: "Lecture 08 — Raw vs Trimmed Reads"
subtitle: "FastQC before and after fastp"
intro_text: "FastQC reports are grouped separately for the raw 5,000-pair subset and the fastp-trimmed reads."

module_order:
  - fastqc:
      name: "FastQC — raw"
      path_filters:
        - "fastqc_raw/*"
  - fastqc:
      name: "FastQC — trimmed"
      path_filters:
        - "fastqc_trimmed/*"
  - fastp:
      name: "fastp — trimming summary"


### Run MultiQC

Combine the raw and trimmed FastQC reports plus the fastp summary into one HTML report.

In [9]:
%%bash

cd /content

# data: search here for FastQC and fastp outputs
# -o data/multiqc: output directory
# -n lecture08_raw_vs_trimmed: report filename
# --config data/multiqc_config.yaml: separate raw and trimmed FastQC sections
multiqc \
  data \
  -o data/multiqc \
  -n lecture08_raw_vs_trimmed \
  --config data/multiqc_config.yaml

ls -lh data/multiqc/

total 2.5M
drwxr-xr-x 2 root root 4.0K Oct 25 13:33 lecture08_raw_vs_trimmed_data
-rw-r--r-- 1 root root 2.5M Oct 25 13:33 lecture08_raw_vs_trimmed.html



/// MultiQC 🔍 v1.35

            config | Loading config settings from: data/multiqc_config.yaml
       file_search | Search path: /content/data

            fastqc | Found 2 reports
            fastqc | Found 2 reports
             fastp | Found 1 reports

     write_results | Data        : data/multiqc/lecture08_raw_vs_trimmed_data
     write_results | Report      : data/multiqc/lecture08_raw_vs_trimmed.html
           multiqc | MultiQC complete


### Expected FastQC result

**Raw**

| | R1 | R2 |
|---|---:|---:|
| Reads | 5,000 | 5,000 |
| Length | 150 bp | 150 bp |
| Mean Q | 35.5 | 37.6 |
| Last 15 bp Q | 30.5 | 34.9 |
| Reads w/ adapter | 2.1% | 2.5% |
| GC | 49% | 49% |

**Trimmed**

| | R1 | R2 |
|---|---:|---:|
| Reads | 4,944 | 4,944 |
| Length | 45–150 bp | 45–150 bp |
| Mean Q | 35.6 | 37.8 |
| Last 15 bp Q | 30.6 | 35.1 |
| Reads w/ adapter | 0.02% | 0.02% |
| GC | 49% | 49% |

> **⏸ STOP — return to the slides.**
>
> We read the FastQC/fastp panel next, then the alignment problem. Do not build the index yet — we start it together in section 1.


## 1. Build the index

STAR's index is the expensive step. It builds a **suffix array** over the reference, and sorting that array is what dominates the runtime.

The finished index for this reference is about 1.7 GB.

Three parameters catch people out.

**`--genomeSAindexNbases`** — STAR's default of 14 is sized for the full genome. For this reference it is too large and STAR warns it may seg-fault at the mapping step. The correct value scales with reference length (`log2(len)/2 - 1`), which is **12** here.

**`--sjdbGTFfile`** — this is the RNA-seq part, and leaving it out is the mistake that makes alignment crawl. Without annotated splice junctions STAR has to discover every intron *de novo*, searching large genomic windows for every batch of reads. With the GTF baked into the index those junctions are already known. `--sjdbOverhang` must be one less than the read length — **149** for 150 bp reads.

**`--genomeSAsparseD`** — the memory-for-speed dial. The suffix array is `(1 + 8/sparseD) x genomeLength`, so `1` costs ~1.6 GB and `2` costs ~0.9 GB. Memory is not the scarce resource here, so take the faster `1`.

Expect this to take several minutes. The index is deterministic, so if you need to re-run later, keep `data/star_index/` and skip straight to step 2.


In [19]:
%%bash

cd /content

sparseD=1            # dense suffix array: ~1.6 GB instead of 0.9 GB, and skips the sparse de-sparsify pass. Memory is not the scarce resource, so take the faster option.
saIndexNbases=12     # STAR's recommendation for a 181 Mb genome. The default 14 is too large here and seg-faults at mapping.

STAR --runThreadN 2 \
     --runMode genomeGenerate \
     --genomeDir data/star_index \
     --genomeFastaFiles data/ref/Homo_sapiens.GRCh38.dna.primary_assembly.sub.fa \
     --sjdbGTFfile data/gtf/Homo_sapiens.GRCh38.sub.gtf \
     --sjdbGTFfeatureExon exon \
     --sjdbOverhang 149 \
     --genomeSAsparseD $sparseD \
     --genomeSAindexNbases $saIndexNbases \
     > data/star_index_build.log 2>&1 &

echo "STAR genomeGenerate started (PID $!)."
echo "Detailed log: data/star_index/Log.out"
sleep 2
tail -n 5 data/star_index/Log.out 2>/dev/null || echo "(Log.out not written yet)"
ls -lh data/star_index/


STAR genomeGenerate started (PID 29232).
Detailed log: data/star_index/Log.out
ParametersSolo: --soloCellFilterType CellRanger2.2 filtering parameters:  3000 0.99 10
Finished loading and checking parameters
--genomeDir directory created: data/star_index/
Oct 25 14:17:44 ... starting to generate Genome files
data/ref/Homo_sapiens.GRCh38.dna.primary_assembly.sub.fa : chr # 0  "17" chrStart: 0
total 4.0K
-rw-r--r-- 1 root root 2.5K Oct 25 14:17 Log.out


> **⏸ STOP — the build is running. Return to the slides.**
>
> The build takes ~25 minutes on 2 vCPUs. **Do not wait for it.** We spend exactly that long on the STAR algorithm, then come back to alignment.
>
> To check progress from the **Terminal** panel:
>
> ```bash
> tail -f /content/data/star_index/Log.out
> ```
>
> `Ctrl+C` stops the tail, not the build. The build is finished when `Log.out` prints `done` and `data/star_index/SA` exists.


## 2. Align the reads

STAR loads the index into memory once, then streams the **trimmed** reads through it. Three settings match our production pipeline:

| setting | why |
|---|---|
| `--outSAMtype BAM SortedByCoordinate` | coordinate-sorted output — what Qualimap expects, and it saves a separate samtools sort |
| `--outFilterType BySJout` | keeps alignments consistent with the splice-junction filters; it is **not** a uniqueness filter |
| `--alignSJoverhangMin 8` / `--alignSJDBoverhangMin 1` | require at least 8 spliced bases at novel junctions, 1 at annotated ones |

Alignment memory is fixed by the index (~3.2 GB) and does **not** grow with read count — 5,000 pairs costs the same RAM as 1.5 M, only less time. At roughly 210 pairs per second this takes a minute and a half. We align the trimmed pairs from `data/trimmed/`.


In [22]:
%%bash

cd /content

dataDir=data
indexDir=$dataDir/star_index

if [ ! -f "$indexDir/SA" ]; then
  echo "Index not ready: $indexDir/SA is missing."
  echo "Wait for the section 1 build to finish, then re-run this cell."
  exit 1
fi
alignDir=$dataDir/STAR
fqDir=$dataDir/trimmed
threads=2                         # Colab has 2 vCPU
sjOverhangMin=8                   # drop junctions anchored on fewer than 8 spliced bases
sjdbOverhangMin=1                 # more permissive: 1 spliced base at annotated junctions

r1=$(ls $fqDir | grep -E '_R1\.trim\.fq$' | sed "s|^|$fqDir/|")
r2=$(ls $fqDir | grep -E '_R2\.trim\.fq$' | sed "s|^|$fqDir/|")

echo "R1: $r1"
echo "R2: $r2"

mkdir -p $alignDir

STAR --runThreadN $threads \
     --runMode alignReads \
     --genomeDir $indexDir \
     --readFilesIn $r1 $r2 \
     --outFileNamePrefix $alignDir/sample. \
     --outSAMtype BAM SortedByCoordinate \
     --outFilterType BySJout \
     --outFilterMultimapNmax 20 \
     --alignSJoverhangMin $sjOverhangMin \
     --alignSJDBoverhangMin $sjdbOverhangMin

ls -lh $alignDir/


R1: data/trimmed/SRR8836894_R1.trim.fq
R2: data/trimmed/SRR8836894_R2.trim.fq
	/usr/lib/rna-star/bin/STAR-avx2 --runThreadN 2 --runMode alignReads --genomeDir data/star_index --readFilesIn data/trimmed/SRR8836894_R1.trim.fq data/trimmed/SRR8836894_R2.trim.fq --outFileNamePrefix data/STAR/sample. --outSAMtype BAM SortedByCoordinate --outFilterType BySJout --outFilterMultimapNmax 20 --alignSJoverhangMin 8 --alignSJDBoverhangMin 1
	STAR version: 2.7.11b   compiled: 2024-04-14T23:10:25+00:00 <place not set in Debian package>
Oct 25 15:13:52 ..... started STAR run
Oct 25 15:13:52 ..... loading genome
Oct 25 15:13:56 ..... started mapping
Oct 25 15:15:19 ..... finished mapping
Oct 25 15:15:19 ..... started sorting BAM
Oct 25 15:15:20 ..... finished successfully
total 192K
-rw-r--r-- 1 root root 164K Oct 25 15:15 sample.Aligned.sortedByCoord.out.bam
-rw-r--r-- 1 root root 2.0K Oct 25 15:15 sample.Log.final.out
-rw-r--r-- 1 root root 6.9K Oct 25 15:15 sample.Log.out
-rw-r--r-- 1 root root  364

To watch the alignment while it runs — STAR writes the BAM only at the end, so watch its progress file, not the output directory:

```bash
watch -n 2 'cat /content/data/STAR/sample.Log.progress.out'
```

`Ctrl+C` stops it.

> **⏸ STOP — return to the slides.**
>
> Leave `sample.Log.final.out` open. We read it together, then dissect the BAM.


## 3. Inspect the BAM

The BAM is the evidence. Before trusting the summary, decode one record yourself.

- `samtools view -H` — the header, including the exact `@PG` command.
- `samtools view | head -1` — one record; find FLAG, CIGAR, MAPQ.
- `samtools view | awk '$6 ~ /N/'` — a spliced read.
- `samtools view -c` vs `-F 256` — all records vs primary.


In [24]:
%%bash

cd /content

bam=data/STAR/sample.Aligned.sortedByCoord.out.bam

echo "=== header (first 8 lines) ==="
samtools view -H "$bam" | head -8

echo
echo "=== first record ==="
samtools view "$bam" | head -1

echo
echo "=== spliced reads (CIGAR contains N) ==="
samtools view "$bam" | awk '$6 ~ /N/' | head -3

echo
echo "=== all vs primary ==="
echo "all records:     $(samtools view -c "$bam")"
echo "primary records: $(samtools view -c -F 256 "$bam")"


=== header (first 8 lines) ===
@HD	VN:1.4	SO:coordinate
@SQ	SN:17	LN:83257441
@SQ	SN:21	LN:46709983
@SQ	SN:22	LN:50818468
@PG	ID:STAR	PN:STAR	VN:2.7.11b	CL:/usr/lib/rna-star/bin/STAR-avx2   --runMode alignReads      --runThreadN 2   --genomeDir data/star_index   --readFilesIn data/trimmed/SRR8836894_R1.trim.fq   data/trimmed/SRR8836894_R2.trim.fq      --outFileNamePrefix data/STAR/sample.   --outSAMtype BAM   SortedByCoordinate      --outFilterType BySJout   --outFilterMultimapNmax 20   --alignSJoverhangMin 8   --alignSJDBoverhangMin 1
@PG	ID:samtools	PN:samtools	PP:STAR	VN:1.19.2	CL:samtools view -H data/STAR/sample.Aligned.sortedByCoord.out.bam
@CO	user command line: /usr/lib/rna-star/bin/STAR-avx2 --runThreadN 2 --runMode alignReads --genomeDir data/star_index --readFilesIn data/trimmed/SRR8836894_R1.trim.fq data/trimmed/SRR8836894_R2.trim.fq --outFileNamePrefix data/STAR/sample. --outSAMtype BAM SortedByCoordinate --outFilterType BySJout --outFilterMultimapNmax 20 --alignSJoverhang

> **⏸ STOP — return to the slides.**
>
> We do QC next: `Log.final.out`, the denominator arithmetic, `flagstat`, and `idxstats`. You will need this BAM.


## 4. QC the alignment

Three sources, none of which needs an install:

- **`sample.Log.final.out`** — STAR's own summary: mapping rate, splice junctions, mismatch rates
- **`samtools flagstat`** — mapped / properly paired / duplicates
- **`samtools idxstats`** — reads per contig

`idxstats` is the one that earns its place here. It shows exactly where the reads that did map went, and how many had nowhere to land.

For a real sample you would reach for two more tools:

| tool | what it adds | note |
|---|---|---|
| **Qualimap** | RNA-seq QC: 5′/3′ bias, duplication rate, coverage uniformity, insert size | what our production pipeline uses; not in Colab's `apt` archive, so it needs conda or the official jar |
| **MultiQC** | aggregates STAR, samtools and picard metrics into one HTML report | what our pipeline finishes with |

Qualimap is not installed here: at 5,000 read pairs these metrics are not statistically meaningful. MultiQC is already installed from the setup section.


In [25]:
%%bash

cd /content

bam=data/STAR/sample.Aligned.sortedByCoord.out.bam

# The .bai enables idxstats and region queries.
samtools index "$bam"

echo "=== STAR summary ==="
grep -E 'input reads|Uniquely mapped reads %|mapped to multiple loci|splices: Total|Mismatch rate' data/STAR/sample.Log.final.out

echo
echo "=== flagstat ==="
samtools flagstat "$bam"

echo
echo "=== reads per contig (idxstats) ==="
samtools idxstats "$bam"

echo
echo "=== insert size ==="
samtools stats "$bam" | grep 'insert size'


=== STAR summary ===
                          Number of input reads |	4944
                        Uniquely mapped reads % |	11.45%
                       Number of splices: Total |	452
                      Mismatch rate per base, % |	1.39%
        Number of reads mapped to multiple loci |	63
             % of reads mapped to multiple loci |	1.27%

=== flagstat ===
1502 + 0 in total (QC-passed reads + QC-failed reads)
1258 + 0 primary
244 + 0 secondary
0 + 0 supplementary
0 + 0 duplicates
0 + 0 primary duplicates
1502 + 0 mapped (100.00% : N/A)
1258 + 0 primary mapped (100.00% : N/A)
1258 + 0 paired in sequencing
629 + 0 read1
629 + 0 read2
1258 + 0 properly paired (100.00% : N/A)
1258 + 0 with itself and mate mapped
0 + 0 singletons (0.00% : N/A)
0 + 0 with mate mapped to a different chr
0 + 0 with mate mapped to a different chr (mapQ>=5)

=== reads per contig (idxstats) ===
17	83257441	844	0
21	46709983	384	0
22	50818468	274	0
*	0	0	0

=== insert size ===
SN	insert size average:	12

### Count the genes

`featureCounts` intersects each fragment with the exons in the GTF and assigns it to a `gene_id`.

- `-p --countReadPairs` — paired-end, count fragments not mates.
- `-s 0` — unstranded. If the library is stranded, `1` or `2`; the wrong value halves the counts.
- `-t exon -g gene_id` — count exons, aggregate by gene.

Read the `.summary` file, not just the matrix. `Assigned` is the numerator; everything else is a diagnosis.


In [26]:
%%bash

cd /content

bam=data/STAR/sample.Aligned.sortedByCoord.out.bam

featureCounts -p --countReadPairs -T 2 -s 0 \
  -t exon -g gene_id \
  -a data/gtf/Homo_sapiens.GRCh38.sub.gtf \
  -o data/STAR/counts.txt \
  "$bam"

echo
echo "=== featureCounts summary ==="
cat data/STAR/counts.txt.summary



=== featureCounts summary ===
Status	data/STAR/sample.Aligned.sortedByCoord.out.bam
Assigned	455
Unassigned_Unmapped	0
Unassigned_Read_Type	0
Unassigned_Singleton	0
Unassigned_MappingQuality	0
Unassigned_Chimera	0
Unassigned_FragmentLength	0
Unassigned_Duplicate	0
Unassigned_MultiMapping	185
Unassigned_Secondary	0
Unassigned_NonSplit	0
Unassigned_NoFeatures	36
Unassigned_Overlapping_Length	0
Unassigned_Ambiguity	75



        ==========     _____ _    _ ____  _____  ______          _____  
        =====         / ____| |  | |  _ \|  __ \|  ____|   /\   |  __ \ 
          =====      | (___ | |  | | |_) | |__) | |__     /  \  | |  | |
            ====      \___ \| |  | |  _ <|  _  /|  __|   / /\ \ | |  | |
              ====    ____) | |__| | |_) | | \ \| |____ / ____ \| |__| |
        ==========   |_____/ \____/|____/|_|  \_\______/_/    \_\_____/
	  v2.0.6

//========================== featureCounts setting ===========================\\
||                                                                            ||
||             Input files : 1 BAM file                                       ||
||                                                                            ||
||                           sample.Aligned.sortedByCoord.out.bam             ||
||                                                                            ||
||             Output file : counts.txt                          

## Appendix — Benchmarking and reduced-reference disclaimer

> **Disclaimer.** This Colab is a benchmark run, not a full production alignment. The FASTQ is a 5,000-pair subset and the reference is reduced to GRCh38 chr17, chr21, chr22 (180.8 Mb, Ensembl release 115). The numbers below describe this reduced setup, not a whole-genome run.

**The constraint is compute, not memory.** A whole-genome STAR index needs ~28 GB of RAM — but the reason it cannot run here is just as much the **hours of CPU** it takes to build on two vCPUs. Cutting the reference to three chromosomes shrinks index time, index size and peak RAM *together*:

| | 3 chromosomes | whole genome |
|---|---|---|
| index build (16 threads) | **77 s** | hours |
| index on disk | **1.7 GB** | ~28 GB |
| peak RAM | **4.1 GB** | ~28 GB |

**A second limit, and it is not memory either** — our reads come from the whole transcriptome, but only 3 of 24 chromosomes are indexed. Roughly **90% of reads cannot map**, and every unmappable read costs a full search before STAR discards it. Throughput is ~210 read pairs/second, which is why we align 5,000 pairs rather than the full 1.5 M. Expect a mapping rate near **11%**, not 90% — that is correct for a reduced reference, not a failure.

### Reproduce the arithmetic

Before you believe the BAM, make the numbers close:

```text
5,000 input fragments
  x (11.06% unique + 1.24% multi) = 615 fragments
  x 2 mates                        = 1,230 primary records
  + 242 secondary                  = 1,472 records
```

Then check the MAPQ distribution:

```bash
samtools view data/STAR/sample.Aligned.sortedByCoord.out.bam | cut -f5 | sort -n | uniq -c
```

> **⏸ STOP — done.**
>
> Homework: decode one spliced record, explain why the expected mapping rate is a property of this reduced reference, and bring one QC number you would report in a methods section.
